In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import Ridge

# 1. Dynamically find the dataset path in Kaggle's input directory
base_path = ""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train.csv':
            base_path = dirname
            break
    if base_path:
        break

print(f"Loading data from: {base_path}")
train_df = pd.read_csv(os.path.join(base_path, 'train.csv'))
valid_df = pd.read_csv(os.path.join(base_path, 'valid.csv'))
test_df = pd.read_csv(os.path.join(base_path, 'test.csv'))

# THE GOLDEN TRICK: Combine Train and Valid sets for maximum temporal accuracy
print("Combining train and valid datasets...")
combined_df = pd.concat([train_df, valid_df], ignore_index=True)

# 2. Define Feature Groups
num_features = [f'x{i}' for i in range(1, 21)] + [f'z{i}' for i in range(1, 9)] + [f'm{i}' for i in range(1, 7)]
cat_features = ['dept', 'building_type', 'sensor_class']
time_features = ['dow', 'month_phase', 'holiday_flag']
lag_features = ['y_lag1', 'y_lag7']

# Helper function for Preprocessing
def get_preprocessor(numeric_cols, categorical_cols, robust=False):
    scaler = RobustScaler() if robust else StandardScaler()
    return ColumnTransformer(transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', scaler)]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])

predictions = []

# ==========================================
# PART A: Correlated Linear Regression
# ==========================================
print("\n--- Training Part A (Ridge) ---")
train_a = combined_df[combined_df['part'] == 'A']
test_a = test_df[test_df['part'] == 'A']

X_train_a, y_train_a = train_a[num_features + cat_features], train_a['y']
X_test_a = test_a[num_features + cat_features]

model_a = Pipeline(steps=[
    ('preprocessor', get_preprocessor(num_features, cat_features)),
    ('regressor', Ridge(alpha=10.0))
])
model_a.fit(X_train_a, y_train_a)
predictions.append(pd.DataFrame({'row_id': test_a['row_id'], 'y': model_a.predict(X_test_a)}))

# ==========================================
# PART B: Localized Nonlinear Structure
# Strategy: Polynomial Regression (Degree 2) to satisfy the professor
# ==========================================
print("--- Training Part B (Polynomial + Ridge) ---")
train_b = combined_df[combined_df['part'] == 'B']
test_b = test_df[test_df['part'] == 'B']

X_train_b, y_train_b = train_b[num_features + cat_features], train_b['y']
X_test_b = test_b[num_features + cat_features]

model_b = Pipeline(steps=[
    ('preprocessor', get_preprocessor(num_features, cat_features)),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)), 
    ('regressor', Ridge(alpha=10.0))
])
model_b.fit(X_train_b, y_train_b)
predictions.append(pd.DataFrame({'row_id': test_b['row_id'], 'y': model_b.predict(X_test_b)}))

# ==========================================
# PART C: Panel Forecasting with Lags
# ==========================================
print("--- Training Part C (Linear Forecasting) ---")
train_c = combined_df[combined_df['part'] == 'C']
test_c = test_df[test_df['part'] == 'C']
all_num_c = num_features + time_features + lag_features

X_train_c, y_train_c = train_c[all_num_c + cat_features], train_c['y']
X_test_c = test_c[all_num_c + cat_features]

model_c = Pipeline(steps=[
    ('preprocessor', get_preprocessor(all_num_c, cat_features)),
    ('regressor', Ridge(alpha=1.0))
])
model_c.fit(X_train_c, y_train_c)
predictions.append(pd.DataFrame({'row_id': test_c['row_id'], 'y': model_c.predict(X_test_c)}))

# ==========================================
# PART D: Distribution Shift & Outliers
# ==========================================
print("--- Training Part D (Robust Linear Forecasting) ---")
train_d = combined_df[combined_df['part'] == 'D']
test_d = test_df[test_df['part'] == 'D']
all_num_d = num_features + time_features + lag_features

X_train_d, y_train_d = train_d[all_num_d + cat_features], train_d['y']
X_test_d = test_d[all_num_d + cat_features]

model_d = Pipeline(steps=[
    ('preprocessor', get_preprocessor(all_num_d, cat_features, robust=True)),
    ('regressor', Ridge(alpha=10.0)) 
])
model_d.fit(X_train_d, y_train_d)
predictions.append(pd.DataFrame({'row_id': test_d['row_id'], 'y': model_d.predict(X_test_d)}))

# ==========================================
# FINAL SUBMISSION COMPILATION
# ==========================================
print("\nCompiling final submission...")
all_preds = pd.concat(predictions)

final_submission = all_preds.set_index('row_id').reindex(test_df['row_id']).reset_index()
final_submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Success! File saved to /kaggle/working/submission.csv")

Loading data from: /kaggle/input/competitions/ell-409-regression-and-forecasting-under-distribution-shift
Combining train and valid datasets...

--- Training Part A (Ridge) ---
--- Training Part B (Polynomial + Ridge) ---
--- Training Part C (Linear Forecasting) ---
--- Training Part D (Robust Linear Forecasting) ---

Compiling final submission...
Success! File saved to /kaggle/working/submission.csv
